# 1 ENV & LIBs

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
!pip install -q ultralytics tensorflow opencv-python numpy Pillow pyyaml matplotlib seaborn onnx onnxruntime

In [ ]:
import os
import yaml
import shutil
import torch
import numpy as np
from ultralytics import YOLO

# 2 PREPROCESSING & YAML CONFIG

In [ ]:
MAIN_FOLDER_PATH = r'D:\Learning\HK2-(2024-2025)\CE340_AI_in_Embed\Final_Project'
SVRDD_DATASET_PATH = os.path.join(MAIN_FOLDER_PATH, 'SVRDD_YOLO')
WORKING_DIR = os.path.join(MAIN_FOLDER_PATH, 'SVRDD_YOLO_Workspace')
os.makedirs(WORKING_DIR, exist_ok=True)

In [ ]:
original_files = ["train.txt", "val.txt", "test.txt"]
output_file_names = ["train_absolute.txt", "val_absolute.txt", "test_absolute.txt"]

for original_file, output_file in zip(original_files, output_file_names):
    input_path = os.path.join(SVRDD_DATASET_PATH, original_file)
    output_path = os.path.join(SVRDD_DATASET_PATH, output_file)
    if not os.path.exists(input_path):
        continue
    with open(input_path, 'r', encoding='utf-8') as infile, open(output_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            stripped = line.strip().replace("\\", "/")
            if stripped:
                abs_path = os.path.normpath(os.path.join(SVRDD_DATASET_PATH, stripped))
                outfile.write(abs_path + '\n')

In [ ]:
data_yaml_content = {
    'path': SVRDD_DATASET_PATH,
    'train': 'train_absolute.txt',
    'val': 'val_absolute.txt',
    'test': 'test_absolute.txt',
    'names': {
        0: 'Alligator Crack', 1: 'Longitudinal Crack', 2: 'Longitudinal Patch',
        3: 'Manhole Cover', 4: 'Pothole', 5: 'Transverse Crack', 6: 'Transverse Patch'
    }
}
yaml_path = os.path.join(WORKING_DIR, 'svrdd.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

# 3 

### TRAIN


In [ ]:
model = YOLO('yolov8n.pt')
training_runs_path = os.path.join(WORKING_DIR, 'training_runs_archive')

results = model.train(
    data=yaml_path, epochs=50, imgsz=640, batch=4, workers=0,
    name='yolov8n_svrdd_run', patience=7, project=training_runs_path
)
best_model_path = os.path.join(results.save_dir, 'weights', 'best.pt')
print(f'Best model saved at: {best_model_path}')

### Evaluation


In [ ]:
best_model = YOLO(best_model_path)
test_metrics = best_model.val(data=yaml_path, split='test', imgsz=320)
print(f'mAP50-95: {test_metrics.box.map:.4f}')
print(f'mAP50:    {test_metrics.box.map50:.4f}')

### Convert

In [ ]:
tflite_path = best_model.export(format='tflite', imgsz=320)
print(f'Exported TFLite model: {tflite_path}')